# Selic — meta e over, acumulados em 12 meses

Motor de coleta e publicação das **duas** séries da família `selic`. As duas vêm do mesmo
lugar — a API de Dados Abertos do Banco Central (SGS) — e as duas são derivadas: o SGS não
publica acumulado em 12 meses de Selic em nenhuma das duas bases.

| série | fonte (SGS) | unidade da fonte | definição publicada |
|---|---|---|---|
| `selic-over-acum-12m` | **11** — Selic, % ao dia | % a.d. | taxa efetiva Selic acumulada nos 12 meses de calendário encerrados no mês de referência |
| `selic-meta-acum-12m` | **432** — meta Selic definida pelo Copom | % a.a. | taxa acumulada em 12 meses equivalente à trajetória da meta no período |

- **Endpoints:**
  `https://api.bcb.gov.br/dados/serie/bcdata.sgs.11/dados?formato=json` ·
  `https://api.bcb.gov.br/dados/serie/bcdata.sgs.432/dados?formato=json`
- **Unidade publicada:** `percentual` (`%`, 2 casas), `eixo: percentual`,
  `medida: acumulado-12m`, `granularidade: mensal`.
- **Fórmula do over:** com o fator diário `f_d = 1 + i_d/100` e o fator do mês
  `F_m = ∏_{d ∈ m} f_d`,
  `V_t = (∏_{k=t−11}^{t} F_k − 1) × 100`. Janela de **12 meses de calendário**, nunca
  "365 linhas".
- **Fórmula da meta:** com `m_m` = média das metas anualizadas em vigor no mês `m`,
  `V_t = (∏_{k=t−11}^{t} (1 + m_k/100)^{1/12} − 1) × 100` — capitalização das taxas
  mensais equivalentes, ou seja, a média geométrica das 12 metas.
- **Período publicado:** `selic-over-acum-12m` de `1995-12-01`; `selic-meta-acum-12m` de
  `2000-03-01`; ambas até o último mês fechado.
- **Arquivos gerados:** `indicadores/selic/selic-over-acum-12m.json` e
  `indicadores/selic/selic-meta-acum-12m.json`, ambos com `versao: 1`.

## Decisão metodológica

**Não existe série oficial pronta para nenhuma das duas.** O SGS publica a Selic diária (11),
a anualizada base 252 (1178), a acumulada no mês (4390, % a.m.) e a acumulada no mês
anualizada (4189, % a.a.) — nenhuma acumulada em 12 meses. Para a meta, o SGS publica apenas
a série 432 (meta anualizada do Copom, desde 05/03/1999); não há acumulado publicado.

**Over.** A série antiga já era o produto dos fatores diários em janelas de 12 meses de
calendário e reproduz o número que o mercado publica (out/2025 = 13,71%; set/2026 = 14,47%).
O defeito era outro: o último ponto saía de um mês ainda aberto (out/2025 publicado como
12,77%, contra 13,71% com o mês fechado), e havia um segundo pipeline morto no notebook
antigo (`rolling(365)` sobre linhas diárias) que nunca chegou ao JSON. Aqui fica **um** caminho:
janela de calendário, meses fechados, série inteira recalculada.

**Meta.** A série antiga **não era um acumulado**: era a média aritmética das metas diárias
anualizadas dentro de cada mês — um nível, não uma taxa do período (jan/2000 = 19,00%, que é
a meta em vigor, não o acumulado de 12 meses). A meta é uma taxa anualizada; acumulá-la em 12
meses exige uma convenção. A adotada aqui converte cada meta mensal em taxa mensal equivalente
(`(1 + m/100)^{1/12}`) e capitaliza as 12 — a mesma convenção base 252 que o BCB usa para
anualizar a Selic diária (252/12 = 21 dias úteis por mês). Propriedade que a define: com a
meta constante nos 12 meses, o acumulado é **exatamente** a meta (jul/2016 = 14,25%).

## Execução sem sobrescrever o publicado

`SELIC_SAIDA` redireciona a pasta de saída (padrão: `indicadores/selic`). Em ensaio:

```
SELIC_SAIDA=/caminho/temporario .venv/bin/python -m nbconvert --to notebook --execute --inplace Selic.ipynb
```

## Coleta

Leitura integral a cada execução — nada de atualização incremental, nada de ler o JSON já
publicado para decidir o período. O SGS limita séries diárias a janelas de 10 anos por
requisição, então a coleta é fatiada em décadas e remontada por data.

In [1]:
from datetime import date
from decimal import Decimal, ROUND_HALF_UP
from pathlib import Path
import json
import math
import os
import time

import requests

RAIZ = Path(os.environ.get("DADOS_RAIZ", Path.cwd())).resolve()
PASTA = RAIZ / "indicadores" / "selic" if (RAIZ / "indicadores" / "selic").is_dir() else RAIZ
DESTINO = Path(os.environ.get("SELIC_SAIDA", PASTA))
ARQUIVO_OVER = DESTINO / "selic-over-acum-12m.json"
ARQUIVO_META = DESTINO / "selic-meta-acum-12m.json"

URL = "https://api.bcb.gov.br/dados/serie/bcdata.sgs.{serie}/dados"
SERIE_OVER = 11
SERIE_META = 432
SERIE_CONFERENCIA = 4390
URL_OVER = URL.format(serie=SERIE_OVER) + "?formato=json"
URL_META = URL.format(serie=SERIE_META) + "?formato=json"

# Recorte editorial: mesma janela das demais famílias (IPCA e IGP-M começam em 1995-01).
# A SGS 11 tem dados desde 1986-06; o acumulado de 12 meses começa 11 meses depois do
# início da coleta, em 1995-12.
INICIO_OVER = date(1995, 1, 1)
HOJE = date.today()
PRIMEIRO_DIA_MES_CORRENTE = HOJE.replace(day=1)


def baixar(serie, inicio, tentativas=4):
    # Baixa uma série SGS desde `inicio`, em janelas de 10 anos (limite do SGS para
    # séries diárias), com nova tentativa e espera crescente quando o serviço falha.
    # Devolve [(date, float)] ascendente, sem data repetida.
    leituras = {}
    cursor = inicio
    while cursor <= HOJE:
        fim = min(HOJE, date(cursor.year + 9, 12, 31))
        parametros = {
            "formato": "json",
            "dataInicial": cursor.strftime("%d/%m/%Y"),
            "dataFinal": fim.strftime("%d/%m/%Y"),
        }
        resposta = None
        for tentativa in range(tentativas):
            try:
                resposta = requests.get(
                    URL.format(serie=serie), params=parametros, timeout=60
                )
                if resposta.status_code == 404:
                    break
                resposta.raise_for_status()
                for item in resposta.json():
                    dia, mes, ano = (int(parte) for parte in item["data"].split("/"))
                    leituras[date(ano, mes, dia)] = float(item["valor"])
                break
            except (requests.RequestException, ValueError):
                if tentativa == tentativas - 1:
                    raise
                time.sleep(2 ** tentativa)
        cursor = date(fim.year + 1, 1, 1)
    if not leituras:
        raise ValueError(f"a série SGS {serie} não retornou dados")
    return sorted(leituras.items())


def por_mes(leituras):
    # Agrupa leituras diárias por mês de calendário, em ordem.
    meses = {}
    for dia, valor in leituras:
        meses.setdefault((dia.year, dia.month), []).append(valor)
    return dict(sorted(meses.items()))


def exigir_meses_continuos(meses, rotulo):
    # Falha se faltar um mês no meio da série: lacuna não pode virar janela torta.
    chaves = list(meses)
    for (ano_a, mes_a), (ano_b, mes_b) in zip(chaves, chaves[1:]):
        if (ano_b * 12 + mes_b) - (ano_a * 12 + mes_a) != 1:
            raise ValueError(
                f"lacuna de mês em {rotulo}: {ano_a}-{mes_a:02d} -> {ano_b}-{mes_b:02d}"
            )


bruto_over = [leitura for leitura in baixar(SERIE_OVER, INICIO_OVER) if leitura[0] >= INICIO_OVER]
bruto_meta = baixar(SERIE_META, date(1999, 1, 1))
conferencia_4390 = [
    leitura for leitura in baixar(SERIE_CONFERENCIA, INICIO_OVER) if leitura[0] >= INICIO_OVER
]

print(f"SGS {SERIE_OVER}: {len(bruto_over)} leituras diárias "
      f"({bruto_over[0][0]:%Y-%m-%d} a {bruto_over[-1][0]:%Y-%m-%d})")
print(f"SGS {SERIE_META}: {len(bruto_meta)} leituras diárias "
      f"({bruto_meta[0][0]:%Y-%m-%d} a {bruto_meta[-1][0]:%Y-%m-%d})")
print(f"SGS {SERIE_CONFERENCIA} (conferência mensal): {len(conferencia_4390)} leituras")
print(f"Pasta de saída: {DESTINO}")

SGS 11: 7990 leituras diárias (1995-01-02 a 2026-10-30)
SGS 432: 10104 leituras diárias (1999-03-05 a 2026-11-01)
SGS 4390 (conferência mensal): 382 leituras
Pasta de saída: /Users/lucas/Workspace/personal/brasilemnumeros/dados/indicadores/selic


## Selic over — composição em janela de 12 meses de calendário

Cada mês de calendário vira um fator `F_m = ∏(1 + i_d/100)` sobre **todos** os dias
publicados naquele mês. O acumulado de 12 meses é o produto dos 12 fatores mensais
consecutivos, menos 1. É a mesma conta que o mercado faz a partir da Selic acumulada no mês
(SGS 4390), só que aqui com a precisão cheia da série diária.

Meses ainda abertos não são publicados: a janela de 12 meses só fecha no último dia do mês.

In [2]:
meses_over = por_mes(bruto_over)
exigir_meses_continuos(meses_over, f"SGS {SERIE_OVER}")

fatores_over = {
    chave: math.prod(1 + valor / 100 for valor in valores)
    for chave, valores in meses_over.items()
}


def acumulado_mensal(fatores, casas=12):
    # Produto dos `casas` fatores mensais consecutivos, menos 1, datado no mês de fechamento.
    itens = list(fatores.items())
    pontos = []
    for posicao in range(casas - 1, len(itens)):
        janela = itens[posicao - casas + 1 : posicao + 1]
        fator = math.prod(valor for _, valor in janela)
        (ano, mes), _ = janela[-1]
        pontos.append((date(ano, mes, 1), (fator - 1) * 100))
    return pontos


def arredondar(valor):
    # 2 casas com meio para cima (convenção comercial).
    return float(Decimal(repr(float(valor))).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP))


bruto_over_12m = acumulado_mensal(fatores_over)
over_12m = [(dia, valor) for dia, valor in bruto_over_12m if dia < PRIMEIRO_DIA_MES_CORRENTE]

if not over_12m:
    raise ValueError("nenhum mês fechado no período publicado")

print(f"Janelas completas: {len(bruto_over_12m)} "
      f"({bruto_over_12m[0][0]:%Y-%m} a {bruto_over_12m[-1][0]:%Y-%m})")
print(f"Mês corrente excluído: {HOJE:%Y-%m}")
print(f"Publicado: {len(over_12m)} meses, {over_12m[0][0]:%Y-%m} a {over_12m[-1][0]:%Y-%m}")
print(f"Primeiro: {over_12m[0][0]:%Y-%m} = {over_12m[0][1]:.4f}%")
print(f"Último:   {over_12m[-1][0]:%Y-%m} = {over_12m[-1][1]:.4f}%")

# Conferência 1: o fator mensal da série diária tem de reproduzir a Selic acumulada no mês
# que o BCB publica na SGS 4390 (arredondada a 2 casas).
oficial_4390 = {
    (dia.year, dia.month): valor for dia, valor in conferencia_4390
}
divergencias = [
    (chave, (fator - 1) * 100, oficial_4390[chave])
    for chave, fator in fatores_over.items()
    if chave in oficial_4390 and abs((fator - 1) * 100 - oficial_4390[chave]) > 0.0051
]
print(f"\nFator mensal x SGS {SERIE_CONFERENCIA}: {len(divergencias)} divergências acima de 0,005 p.p. "
      f"em {len(set(fatores_over) & set(oficial_4390))} meses comuns")
assert not divergencias, divergencias[:5]

# Conferência 2: o acumulado também fecha quando composto só da série oficial 4390.
ultimo = over_12m[-1][0]
chaves_janela = [
    (ultimo.year + (ultimo.month - 1 - deslocamento) // 12,
     (ultimo.month - 1 - deslocamento) % 12 + 1)
    for deslocamento in range(11, -1, -1)
]
fator_oficial = math.prod(1 + oficial_4390[chave] / 100 for chave in chaves_janela)
print(f"Última janela pela SGS {SERIE_CONFERENCIA}: {(fator_oficial - 1) * 100:.4f}% "
      f"x {over_12m[-1][1]:.4f}% pela SGS {SERIE_OVER}")
assert abs((fator_oficial - 1) * 100 - over_12m[-1][1]) < 0.02

Janelas completas: 371 (1995-12 a 2026-10)
Mês corrente excluído: 2026-11
Publicado: 371 meses, 1995-12 a 2026-10
Primeiro: 1995-12 = 53.0872%
Último:   2026-10 = 14.2438%

Fator mensal x SGS 4390: 0 divergências acima de 0,005 p.p. em 382 meses comuns
Última janela pela SGS 4390: 14.2315% x 14.2438% pela SGS 11


## Selic meta — capitalização das taxas mensais equivalentes

A meta é uma taxa **anualizada**: `15%` significa `15% ao ano`, não `15% no mês`. Acumulá-la
em 12 meses exige converter cada meta mensal em taxa mensal equivalente,
`(1 + m/100)^{1/12}`, e capitalizar 12 meses:

`V_t = (∏_{k=t−11}^{t} (1 + m_k/100)^{1/12} − 1) × 100`

Propriedade de controle: se a meta não muda nos 12 meses, `V_t` é exatamente a meta
(`14,25%` em jul/2016). A média aritmética das metas daria o mesmo número nesse caso, mas
erra quando a meta muda; a composição é a taxa efetiva do período.

O primeiro mês da SGS 432 é março/1999 **parcial** (a série começa em 05/03/1999). Ele fica
fora da média mensal, então a primeira janela de 12 meses completos fecha em março/2000.

In [3]:
meses_meta = por_mes(bruto_meta)
exigir_meses_continuos(meses_meta, f"SGS {SERIE_META}")

primeiro_mes_meta = next(iter(meses_meta))
if primeiro_mes_meta != (1999, 3):
    raise ValueError(f"primeiro mês inesperado na SGS {SERIE_META}: {primeiro_mes_meta}")
dias_no_primeiro_mes = len(meses_meta.pop(primeiro_mes_meta))
print(f"Mês parcial descartado: {primeiro_mes_meta[0]}-{primeiro_mes_meta[1]:02d} "
      f"({dias_no_primeiro_mes} dias; a série começa em 05/03/1999)")

meta_mensal = {chave: sum(valores) / len(valores) for chave, valores in meses_meta.items()}
fatores_meta = {chave: (1 + valor / 100) ** (1 / 12) for chave, valor in meta_mensal.items()}

bruto_meta_12m = acumulado_mensal(fatores_meta)
meta_12m = [(dia, valor) for dia, valor in bruto_meta_12m if dia < PRIMEIRO_DIA_MES_CORRENTE]

if not meta_12m:
    raise ValueError("nenhum mês fechado no período publicado")

print(f"Janelas completas: {len(bruto_meta_12m)} "
      f"({bruto_meta_12m[0][0]:%Y-%m} a {bruto_meta_12m[-1][0]:%Y-%m})")
print(f"Publicado: {len(meta_12m)} meses, {meta_12m[0][0]:%Y-%m} a {meta_12m[-1][0]:%Y-%m}")
print(f"Primeiro: {meta_12m[0][0]:%Y-%m} = {meta_12m[0][1]:.4f}%")
print(f"Último:   {meta_12m[-1][0]:%Y-%m} = {meta_12m[-1][1]:.4f}%")

# Controle: janela inteira com a meta constante em 14,25% (ago/2015 a jul/2016).
alvo_controle = date(2016, 7, 1)
chaves_controle = [
    (alvo_controle.year + (alvo_controle.month - 1 - deslocamento) // 12,
     (alvo_controle.month - 1 - deslocamento) % 12 + 1)
    for deslocamento in range(11, -1, -1)
]
assert all(meta_mensal[chave] == 14.25 for chave in chaves_controle), [
    (chave, meta_mensal[chave]) for chave in chaves_controle
]
valor_controle = dict(meta_12m)[alvo_controle]
print(f"Controle (meta constante de 14,25%): {alvo_controle:%Y-%m} = {valor_controle:.10f}%")
assert abs(valor_controle - 14.25) < 1e-9

Mês parcial descartado: 1999-03 (27 dias; a série começa em 05/03/1999)
Janelas completas: 321 (2000-03 a 2026-11)
Publicado: 320 meses, 2000-03 a 2026-10
Primeiro: 2000-03 = 21.5668%
Último:   2026-10 = 14.5368%
Controle (meta constante de 14,25%): 2016-07 = 14.2500000000%


## Escrita dos JSON do contrato

Cada arquivo é gravado inteiro — `versao`, `id`, `unidade`, `eixo`, `medida`,
`granularidade`, `fonte`, `urlFonte` e `dados` — sem ler nem preservar nada do que está
publicado. Pontos no formato `{"data": "YYYY-MM-DD", "valor": ...}`, arredondados a 2 casas.

In [4]:
def montar_serie(id_serie, fonte_url, pontos):
    return {
        "versao": 1,
        "id": id_serie,
        "unidade": {"tipo": "percentual", "simbolo": "%", "casas": 2},
        "eixo": "percentual",
        "medida": "acumulado-12m",
        "granularidade": "mensal",
        "fonte": "Banco Central do Brasil",
        "urlFonte": fonte_url,
        "dados": [
            {"data": dia.strftime("%Y-%m-%d"), "valor": arredondar(valor)}
            for dia, valor in pontos
        ],
    }


serie_over = montar_serie("selic-over-acum-12m", URL_OVER, over_12m)
serie_meta = montar_serie("selic-meta-acum-12m", URL_META, meta_12m)

DESTINO.mkdir(parents=True, exist_ok=True)
for arquivo, serie in ((ARQUIVO_OVER, serie_over), (ARQUIVO_META, serie_meta)):
    with arquivo.open("w", encoding="utf-8") as destino:
        json.dump(serie, destino, ensure_ascii=False, indent=2)
        destino.write("\n")
    print(f"{len(serie['dados'])} pontos -> {arquivo}")
    print(f"  {serie['dados'][0]}")
    print(f"  {serie['dados'][-1]}")

371 pontos -> /Users/lucas/Workspace/personal/brasilemnumeros/dados/indicadores/selic/selic-over-acum-12m.json
  {'data': '1995-12-01', 'valor': 53.09}
  {'data': '2026-10-01', 'valor': 14.24}
320 pontos -> /Users/lucas/Workspace/personal/brasilemnumeros/dados/indicadores/selic/selic-meta-acum-12m.json
  {'data': '2000-03-01', 'valor': 21.57}
  {'data': '2026-10-01', 'valor': 14.54}


## Validação inline

A célula falha se o contrato, a ordem, a unicidade e a continuidade dos meses, a finitude dos
valores, a composição ou a conferência contra a fonte forem violados. Os números citados em
`docs/metodologia.md` e a comparação com valores publicados de forma independente também são
conferidos aqui.

In [5]:
CHAVES_RAIZ = {
    "versao", "id", "unidade", "eixo", "medida", "granularidade", "fonte", "urlFonte", "dados",
}
ESPERADO = {
    "selic-over-acum-12m": (URL_OVER, "1995-12-01"),
    "selic-meta-acum-12m": (URL_META, "2000-03-01"),
}


def validar(serie, arquivo):
    assert set(serie) == CHAVES_RAIZ, sorted(set(serie) ^ CHAVES_RAIZ)
    assert serie["versao"] == 1, serie["versao"]
    assert serie["id"] in ESPERADO, serie["id"]
    assert serie["unidade"] == {"tipo": "percentual", "simbolo": "%", "casas": 2}, serie["unidade"]
    assert serie["eixo"] == "percentual", serie["eixo"]
    assert serie["medida"] == "acumulado-12m", serie["medida"]
    assert serie["granularidade"] == "mensal", serie["granularidade"]
    assert serie["fonte"] == "Banco Central do Brasil", serie["fonte"]
    url_esperada, inicio_esperado = ESPERADO[serie["id"]]
    assert serie["urlFonte"] == url_esperada, serie["urlFonte"]

    dados = serie["dados"]
    assert dados, "série vazia"
    for ponto in dados:
        assert set(ponto) == {"data", "valor"}, sorted(ponto)
        assert isinstance(ponto["data"], str), ponto
        assert not isinstance(ponto["valor"], bool), ponto
        assert isinstance(ponto["valor"], (int, float)), ponto
        assert math.isfinite(ponto["valor"]), ponto

    dias = [date.fromisoformat(ponto["data"]) for ponto in dados]
    assert all(a < b for a, b in zip(dias, dias[1:])), "datas fora de ordem ou repetidas"
    assert all(dia.day == 1 for dia in dias), "dia diferente de 01"
    assert all(
        (b.year * 12 + b.month) - (a.year * 12 + a.month) == 1 for a, b in zip(dias, dias[1:])
    ), "mês faltando ou repetido"
    assert dados[0]["data"] == inicio_esperado, dados[0]["data"]
    assert dias[-1] < PRIMEIRO_DIA_MES_CORRENTE, dados[-1]["data"]

    with arquivo.open(encoding="utf-8") as origem:
        assert json.load(origem) == serie, "arquivo gravado difere do objeto validado"
    print(f"ok {serie['id']}: {len(dados)} meses, {dados[0]['data']} a {dados[-1]['data']}, "
          f"último {dados[-1]['valor']:.2f}%")
    return {ponto["data"]: ponto["valor"] for ponto in dados}


por_over = validar(serie_over, ARQUIVO_OVER)
por_meta = validar(serie_meta, ARQUIVO_META)

assert len(serie_over["dados"]) == len(over_12m)
assert len(serie_meta["dados"]) == len(meta_12m)

# Os valores gravados são a composição recalculada acima, sem sobra nem falta.
assert serie_over["dados"] == [
    {"data": dia.strftime("%Y-%m-%d"), "valor": arredondar(valor)} for dia, valor in over_12m
]
assert serie_meta["dados"] == [
    {"data": dia.strftime("%Y-%m-%d"), "valor": arredondar(valor)} for dia, valor in meta_12m
]

# --- over: conferência contra valores publicados de forma independente --------------
# Set/2026 e out/2025 aparecem em tabelas de mercado que compõem a SGS 4390; é a mesma conta.
assert por_over["2025-10-01"] == 13.71, por_over["2025-10-01"]
assert por_over["2026-09-01"] == 14.47, por_over["2026-09-01"]
# Choques e datas recentes citados em docs/metodologia.md.
assert por_over["2002-12-01"] == 19.17, por_over["2002-12-01"]
assert por_over["2008-12-01"] == 12.48, por_over["2008-12-01"]
assert por_over["2020-12-01"] == 2.76, por_over["2020-12-01"]
assert por_over["2021-12-01"] == 4.42, por_over["2021-12-01"]
assert por_over["2026-10-01"] == 14.24, por_over["2026-10-01"]

# --- meta: composição, não nível nem soma -------------------------------------------
assert abs(por_meta["2016-07-01"] - 14.25) < 1e-9, por_meta["2016-07-01"]
assert por_meta["2003-12-01"] == 23.48, por_meta["2003-12-01"]
assert por_meta["2002-12-01"] == 19.21, por_meta["2002-12-01"]
assert por_meta["2020-12-01"] == 2.89, por_meta["2020-12-01"]
assert por_meta["2021-12-01"] == 4.50, por_meta["2021-12-01"]
assert por_meta["2025-10-01"] == 13.82, por_meta["2025-10-01"]
assert por_meta["2026-10-01"] == 14.54, por_meta["2026-10-01"]

# Em dez/2003 a meta em vigor era ~17,0%: o acumulado de 12 meses não é o nível do mês.
assert abs(por_meta["2003-12-01"] - round(meta_mensal[(2003, 12)], 2)) > 5, (
    por_meta["2003-12-01"], meta_mensal[(2003, 12)]
)

print("\ntudo certo: over e meta no contrato, janelas de 12 meses de calendário, meses fechados")

ok selic-over-acum-12m: 371 meses, 1995-12-01 a 2026-10-01, último 14.24%
ok selic-meta-acum-12m: 320 meses, 2000-03-01 a 2026-10-01, último 14.54%

tudo certo: over e meta no contrato, janelas de 12 meses de calendário, meses fechados
